In [1]:
from collections import Counter
from pathlib import Path
import json
import pandas as pd

BASE = Path("/home/ss99569/code/video-agent/VSeek-R1/scripts/baselines/VideoTree/prepared/pipeline")
DATASETS = ["mlvu", "videomme","lvb","lvbench", "cgbench"]


def majority_vote(xs):
    xs = [x for x in xs if x is not None and x != ""]
    return Counter(xs).most_common(1)[0][0] if xs else None


rows = []
for dataset in DATASETS:
    qa = json.loads((BASE / dataset / "stage3_qa" / "qa.json").read_text())
    items = list(qa["data"].values())

    pass1 = sum(item["pred_runs"][0] == item["truth"] for item in items) / len(items)
    pass4 = sum(any(pred == item["truth"] for pred in item["pred_runs"][:4]) for item in items) / len(items)
    majority4 = sum(majority_vote(item["pred_runs"][:4]) == item["truth"] for item in items) / len(items)
    frames_total = sum(item["sampled_frame_count"] for item in items)
    frames_avg = frames_total / len(items)

    rows.append(
        {
            "dataset": dataset,
            "pass@1": pass1,
            "pass@4": pass4,
            "majority@4": majority4,
            "frames_total": frames_total,
            "frames_avg": frames_avg,
        }
    )

summary = pd.DataFrame(rows)
summary[["pass@1", "pass@4", "majority@4", "frames_avg"]] = summary[["pass@1", "pass@4", "majority@4", "frames_avg"]].round(4)
summary

,dataset,pass@1,pass@4,majority@4,frames_total,frames_avg
0,mlvu,0.4609,0.4816,0.4683,521929,240.0777
1,videomme,0.5633,0.5800,0.5663,455542,168.7193
2,lvb,0.4540,0.4757,0.4510,317660,237.5916
3,lvbench,0.3396,0.3635,0.3415,609212,393.2937
4,cgbench,0.2960,0.3193,0.3003,1146545,382.1817


In [10]:
LENGTH_BINS = [0, 60, 600, 3600, float("inf")]
LENGTH_BIN_LABELS = ["<1m", "1m-10m", "10-60m", "60m+"]


def load_duration_map(dataset):
    duration_map = {}
    for shard_dir in sorted((BASE / dataset / "_parallel_shards").glob("shard_*")):
        duration_map.update(json.loads((shard_dir / "duration.json").read_text()))
    return {str(k): float(v) for k, v in duration_map.items()}


rows = []
for dataset in DATASETS:
    qa = json.loads((BASE / dataset / "stage3_qa" / "qa.json").read_text())
    duration_map = load_duration_map(dataset)

    for item in qa["data"].values():
        uid = str(item["uid"])
        if uid not in duration_map:
            continue
        rows.append(
            {
                "dataset": dataset,
                "length_bucket": pd.cut(
                    [duration_map[uid]],
                    bins=LENGTH_BINS,
                    labels=LENGTH_BIN_LABELS,
                    right=False,
                    include_lowest=True,
                )[0],
                "is_correct": majority_vote(item["pred_runs"][:4]) == item["truth"],
            }
        )

acc_by_length = (
    pd.DataFrame(rows)
    .groupby(["dataset", "length_bucket"], observed=True)["is_correct"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "accuracy", "count": "num_questions"})
)
acc_by_length["length_bucket"] = pd.Categorical(
    acc_by_length["length_bucket"], categories=LENGTH_BIN_LABELS, ordered=True
)
acc_by_length["accuracy"] = acc_by_length["accuracy"].round(4)
acc_by_length = acc_by_length.sort_values(["dataset", "length_bucket"]).reset_index(drop=True)
display(acc_by_length)

for dataset in DATASETS:
    print(dataset)
    dataset_df = (
        acc_by_length[acc_by_length["dataset"] == dataset][["length_bucket", "accuracy", "num_questions"]]
        .set_index("length_bucket")
        .reindex(LENGTH_BIN_LABELS)
    )
    display(dataset_df)
    print(f"Per-bucket performance ({dataset}):")
    for bucket in LENGTH_BIN_LABELS:
        if bucket not in dataset_df.index:
            continue
        row = dataset_df.loc[bucket]
        if pd.isna(row["accuracy"]):
            continue
        print(f"  {bucket}: accuracy={row['accuracy']:.4f} (n={int(row['num_questions'])})")

,dataset,length_bucket,accuracy,num_questions
0,lvb,<1m,0.4721,341
1,lvb,1m-10m,0.4744,430
2,lvb,10-60m,0.4205,566
3,lvbench,10-60m,0.3745,729
4,lvbench,60m+,0.3122,820
5,mlvu,1m-10m,0.4471,1682
6,mlvu,10-60m,0.5533,450
7,mlvu,60m+,0.4048,42
8,videomme,<1m,0.6096,228
9,videomme,1m-10m,0.5849,1272


mlvu


,accuracy,num_questions
length_bucket,,
<1m,NaN,NaN
1m-10m,0.4471,1682.0
10-60m,0.5533,450.0
60m+,0.4048,42.0


Per-bucket performance (mlvu):
  1m-10m: accuracy=0.4471 (n=1682)
  10-60m: accuracy=0.5533 (n=450)
  60m+: accuracy=0.4048 (n=42)
videomme


,accuracy,num_questions
length_bucket,,
<1m,0.6096,228.0
1m-10m,0.5849,1272.0
10-60m,0.5383,1200.0
60m+,NaN,NaN


Per-bucket performance (videomme):
  <1m: accuracy=0.6096 (n=228)
  1m-10m: accuracy=0.5849 (n=1272)
  10-60m: accuracy=0.5383 (n=1200)
lvb


,accuracy,num_questions
length_bucket,,
<1m,0.4721,341.0
1m-10m,0.4744,430.0
10-60m,0.4205,566.0
60m+,NaN,NaN


Per-bucket performance (lvb):
  <1m: accuracy=0.4721 (n=341)
  1m-10m: accuracy=0.4744 (n=430)
  10-60m: accuracy=0.4205 (n=566)
lvbench


,accuracy,num_questions
length_bucket,,
<1m,NaN,NaN
1m-10m,NaN,NaN
10-60m,0.3745,729.0
60m+,0.3122,820.0


Per-bucket performance (lvbench):
  10-60m: accuracy=0.3745 (n=729)
  60m+: accuracy=0.3122 (n=820)


In [11]:
frame_rows = []
for dataset in DATASETS:
    qa = json.loads((BASE / dataset / "stage3_qa" / "qa.json").read_text())
    duration_map = load_duration_map(dataset)

    for item in qa["data"].values():
        uid = str(item["uid"])
        if uid not in duration_map:
            continue
        frame_rows.append(
            {
                "dataset": dataset,
                "length_bucket": pd.cut(
                    [duration_map[uid]],
                    bins=LENGTH_BINS,
                    labels=LENGTH_BIN_LABELS,
                    right=False,
                    include_lowest=True,
                )[0],
                "frames_used": item["sampled_frame_count"],
            }
        )

frames_by_length = (
    pd.DataFrame(frame_rows)
    .groupby(["dataset", "length_bucket"], observed=True)["frames_used"]
    .agg(["mean", "sum", "count"])
    .reset_index()
    .rename(columns={"mean": "avg_frames", "sum": "total_frames", "count": "num_questions"})
)
frames_by_length["length_bucket"] = pd.Categorical(
    frames_by_length["length_bucket"], categories=LENGTH_BIN_LABELS, ordered=True
)
frames_by_length[["avg_frames"]] = frames_by_length[["avg_frames"]].round(4)
frames_by_length = frames_by_length.sort_values(["dataset", "length_bucket"]).reset_index(drop=True)
display(frames_by_length)

for dataset in DATASETS:
    print(dataset)
    dataset_df = (
        frames_by_length[frames_by_length["dataset"] == dataset][["length_bucket", "avg_frames", "total_frames", "num_questions"]]
        .set_index("length_bucket")
        .reindex(LENGTH_BIN_LABELS)
    )
    display(dataset_df)
    print(f"Frames per video bucket ({dataset}):")
    for bucket in LENGTH_BIN_LABELS:
        if bucket not in dataset_df.index:
            continue
        row = dataset_df.loc[bucket]
        if pd.isna(row["avg_frames"]):
            continue
        print(
            f"  {bucket}: avg_frames={row['avg_frames']:.4f}, total_frames={int(row['total_frames'])} (n={int(row['num_questions'])})"
        )

,dataset,length_bucket,avg_frames,total_frames,num_questions
0,lvb,<1m,13.7331,4683,341
1,lvb,1m-10m,205.3837,88315,430
2,lvb,10-60m,396.9293,224662,566
3,lvbench,10-60m,399.0370,290898,729
4,lvbench,60m+,388.1878,318314,820
5,mlvu,1m-10m,213.7051,359452,1682
6,mlvu,10-60m,323.5022,145576,450
7,mlvu,60m+,402.4048,16901,42
8,videomme,<1m,26.4825,6038,228
9,videomme,1m-10m,111.3270,141608,1272


mlvu


,avg_frames,total_frames,num_questions
length_bucket,,,
<1m,NaN,NaN,NaN
1m-10m,213.7051,359452.0,1682.0
10-60m,323.5022,145576.0,450.0
60m+,402.4048,16901.0,42.0


Frames per video bucket (mlvu):
  1m-10m: avg_frames=213.7051, total_frames=359452 (n=1682)
  10-60m: avg_frames=323.5022, total_frames=145576 (n=450)
  60m+: avg_frames=402.4048, total_frames=16901 (n=42)
videomme


,avg_frames,total_frames,num_questions
length_bucket,,,
<1m,26.4825,6038.0,228.0
1m-10m,111.3270,141608.0,1272.0
10-60m,256.5800,307896.0,1200.0
60m+,NaN,NaN,NaN


Frames per video bucket (videomme):
  <1m: avg_frames=26.4825, total_frames=6038 (n=228)
  1m-10m: avg_frames=111.3270, total_frames=141608 (n=1272)
  10-60m: avg_frames=256.5800, total_frames=307896 (n=1200)
lvb


,avg_frames,total_frames,num_questions
length_bucket,,,
<1m,13.7331,4683.0,341.0
1m-10m,205.3837,88315.0,430.0
10-60m,396.9293,224662.0,566.0
60m+,NaN,NaN,NaN


Frames per video bucket (lvb):
  <1m: avg_frames=13.7331, total_frames=4683 (n=341)
  1m-10m: avg_frames=205.3837, total_frames=88315 (n=430)
  10-60m: avg_frames=396.9293, total_frames=224662 (n=566)
lvbench


,avg_frames,total_frames,num_questions
length_bucket,,,
<1m,NaN,NaN,NaN
1m-10m,NaN,NaN,NaN
10-60m,399.0370,290898.0,729.0
60m+,388.1878,318314.0,820.0


Frames per video bucket (lvbench):
  10-60m: avg_frames=399.0370, total_frames=290898 (n=729)
  60m+: avg_frames=388.1878, total_frames=318314 (n=820)
